# Register Allocation in LLVM — a hands-on tour with `eudsl-llvmpy`

This notebook explains how the **LLVM** compiler infrastructure's register
allocator works by *driving it from Python*. Every idea is illustrated with runnable code against the `eudsl-llvmpy`
bindings (`llvm.mir`), which are a thin hand-written wrapper over LLVM's C++
Machine IR (MIR — the **intermediate representation**, or **IR**, that LLVM uses
after instruction selection) and register-allocation machinery.

We'll build up in layers:

1. The problem: virtual vs physical registers.
2. Machine IR, live-ins, and how to read it.
3. Running a real allocator (`greedy`) and reading the result back.
4. Live intervals and *interference* — the heart of the problem.
5. The allocation framework: writing your own allocator (`select_or_split`).
6. The interference *matrix* and `allocation_order`.
7. Spilling and reloads (what happens under register pressure).
8. Eviction and live-range splitting (what `greedy` actually does).
9. Global allocation: the ILP (integer linear programming) allocators, and why
   spilling is subtle.

> Requires the AArch64 (64-bit ARM) backend linked into the extension and `ortools` (for the
> last section). Both are assumed present here.

## Prologue: reading AArch64 register names (`W` vs `X`)

Throughout this notebook you'll see physical registers written as `W0`, `W1`,
… `W30`. The leading letter tells you the *width* of the register:

- **`X0`–`X30`** — the full **64-bit** register. `X` stands for *extended*
  (a "doubleword").
- **`W0`–`W30`** — the low **32 bits** of that *same* register. In ARM
  terminology a **`W`ord is 32 bits**, so `W0` is the 32-bit view of `X0`.

Two consequences worth keeping in mind:

- `W0` and `X0` are **not** two different registers — `W0` is the bottom half of
  `X0`. AArch64 has 31 general-purpose registers, each with a 32-bit (`W`) and a
  64-bit (`X`) name.
- Writing a `W` register **zeroes the upper 32 bits** of the corresponding `X`
  register (a clean 32→64 zero-extend, unlike x86's partial-register writes).

Our example function adds two `i32` values, so its operands live in the 32-bit
(`W`) view — which is why you'll see `W0`/`W1`, the `$w0` physreg in the printed
MIR, and the register class named **`GPR32`** (the 32-bit general-purpose class).
An `i64` version would instead use `X0`/`X1` and the `GPR64` class.

### Reading opcode names

LLVM's AArch64 opcodes (what you pass to `mf.opcode(...)`) encode the operation,
the operand width, and the operand *form* in the name. For example `ADDWrr`
reads as **`ADD`** + **`W`** (32-bit; `X` = 64-bit) + **`rr`** (both sources are
registers). The suffix distinguishes the addressing forms of the same `ADD`:

| Opcode  | Second operand      | Assembly example            |
|---------|---------------------|-----------------------------|
| `ADDWrr` | register            | `add w0, w1, w2`            |
| `ADDWri` | immediate           | `add w0, w1, #4`            |
| `ADDWrs` | register, shifted   | `add w0, w1, w2, lsl #3`    |
| `ADDWrx` | register, extended  | `add w0, w1, w2, uxtb`      |

Our example adds two `i32` values already in registers, so it uses `ADDWrr`
(and the plain `COPY` and `RET_ReallyLR` pseudo-ops). These names come straight
from LLVM's AArch64 target description.

In [10]:
import llvm
from llvm import ir, jit, mir
from typing import Callable

TRIPLE = "aarch64-unknown-linux-gnu"
assert "aarch64" in llvm.jit.registered_targets(), "AArch64 backend not linked"
print("llvm loaded; AArch64 target available")

llvm loaded; AArch64 target available


## 1. The problem in one sentence

When the compiler finishes instruction selection it has code that uses an
*unbounded* number of **virtual registers** (temporaries: `%0`, `%1`, …). Real
hardware has a *fixed, small* set of **physical registers** (on AArch64:
`W0`–`W30` for 32-bit **general-purpose registers**, GPRs). **Register allocation** is the pass that maps each
virtual register to a physical register — and, when there aren't enough,
*spills* some values to memory.

Let's make that concrete. First, a helper to name physical registers (the
bindings expose numeric ids; we build a small id→name map for the GPR32 class we
care about).

In [11]:
# The bindings give registers as integer ids. Build id<->name maps for the
# registers we use so the output is readable.
_PHYS_NAMES = ["W%d" % i for i in range(31)] + ["WZR", "WSP"]


def phys_name_map(mf: mir.MachineFunction) -> dict[int, str]:
    m = {}
    for nm in _PHYS_NAMES:
        try:
            m[mf.physreg(nm).id] = nm
        except Exception:
            pass
    return m


def is_virtual(reg_id: int) -> bool:
    # LLVM tags virtual registers with the top bit (0x80000000) set.
    return reg_id >= 0x80000000


def reg_name(reg_id: int, names: dict[int, str] | None = None) -> str:
    # Print virtual registers as %N (N = the low bits), matching the MIR's %0,
    # %1, ...; physical registers print by name (W0, ...) when a map is given.
    if is_virtual(reg_id):
        return f"%{reg_id & 0x7FFFFFFF}"
    return (names or {}).get(reg_id, f"${reg_id}")

## 2. Build a tiny function and read its Machine IR

We hand-build the MIR for something like `int add(int a, int b){ return a+b; }`.
Arguments arrive in the physical registers `W0`/`W1` (the AArch64 calling
convention), so the function is `COPY`-ed into virtual registers, added, and the
result `COPY`-ed back into `W0` to return.

Watch the printed MIR: `$w0` is a **physical** register (dollar sign), `%0` is a
**virtual** register (percent), and `liveins:` records which physregs are live
on entry.

In [12]:
# The MIR we build below (before register allocation) is:
#
#     %0:gpr32 = COPY $w0
#     %1:gpr32 = COPY $w1
#     %2:gpr32 = ADDWrr %0, %1
#     $w0      = COPY %2
#     RET_ReallyLR implicit $w0
#
# After the greedy allocator assigns registers, all the COPYs coalesce away
# (%0->w0, %1->w1, %2->w0), so the emitted AArch64 assembly is just:
#
#     add    w0, w0, w1
#     ret
#
# (obtained via mmi.emit_object() disassembled with llvm-objdump).
def build_add(
    mmi: mir.MirModule,
) -> tuple[mir.MachineFunction, tuple[mir.Register, mir.Register, mir.Register]]:
    mf = mmi.machine_function("add")
    b = mir.MachineIRBuilder(mf)
    entry = mf.blocks[0]
    gpr32 = mf.reg_class("GPR32")
    w0, w1 = mf.physreg("W0"), mf.physreg("W1")
    entry.add_livein(w0)
    entry.add_livein(w1)
    v0, v1, v2 = (mf.create_vreg(gpr32) for _ in range(3))
    copy = mf.opcode("COPY")
    for dst, src in ((v0, w0), (v1, w1)):
        c = b.build_instr(copy)
        c.add_reg(dst, is_def=True)
        c.add_reg(src)
    add = b.build_instr(mf.opcode("ADDWrr"))
    add.add_reg(v2, is_def=True)
    add.add_reg(v0)
    add.add_reg(v1)
    rc = b.build_instr(copy)
    rc.add_reg(w0, is_def=True)
    rc.add_reg(v2)
    b.build_instr(mf.opcode("RET_ReallyLR")).add_reg(w0, implicit=True)
    for prop in ("IsSSA", "TracksLiveness", "NoPHIs"):
        mf.set_property(getattr(mir.MachineFunctionProperty, prop))
    return mf, (v0, v1, v2)


with ir.Context() as ctx:
    mod = ir.Module("m", ctx)
    tm = jit.TargetMachine(triple=TRIPLE)
    mmi = mir.create_machine_function(mod, tm, "add")
    mf, (v0, v1, v2) = build_add(mmi)
    print(mf)
    print(
        "virtual regs:",
        [reg_name(v.id) for v in (v0, v1, v2)],
        "-> all virtual?",
        all(is_virtual(v.id) for v in (v0, v1, v2)),
    )
    print(
        "W0 id:", mf.physreg("W0").id, "(physical?", mf.physreg("W0").is_physical, ")"
    )

# Machine code for function add: IsSSA, NoPHIs, TracksLiveness

bb.0:
  liveins: $w0, $w1
  %0:gpr32 = COPY $w0
  %1:gpr32 = COPY $w1
  %2:gpr32 = ADDWrr %0:gpr32, %1:gpr32
  $w0 = COPY %2:gpr32
  RET_ReallyLR implicit $w0

# End machine code for function add.


virtual regs: ['%0', '%1', '%2'] -> all virtual? True
W0 id: 208 (physical? True )


### Reflect

- Why are there `COPY`s at all? (Think about *where* the **ABI** (application binary interface) says arguments
  and
  return values must live, versus where the `ADD` can read/write.)
- The MIR is marked `IsSSA` (**static single assignment**). In SSA, every
  virtual register is written exactly
  once. Keep that in mind — it makes the interference structure special (we'll
  return to this).

<details>
<summary><b>Answers</b> (click to expand)</summary>

- **Why the `COPY`s?** The ABI *pins* values to specific physical registers — the
  incoming args in `W0`/`W1`, the return value in `W0` — but the body works in
  **virtual** registers so it stays in SSA and lets the allocator choose
  placement. The `COPY`s bridge those fixed physregs to/from vregs. (They usually
  vanish later via coalescing: recall the emitted asm was just `add w0, w0, w1;
  ret`.)
- **Why SSA matters:** each vreg is defined exactly once, so its lifetime is a
  single value with one def. The interference graph of strict SSA is *chordal*,
  which makes **coloring** easy (polynomial) — so the genuinely hard part of
  allocation moves into *spilling* and *coalescing*. We cash this in at Section 9.

</details>

## 3. Run a real allocator and read the result

The bindings can run LLVM's production **greedy** allocator and hand back the
decisions: `regalloc_assignments(regalloc="greedy")` returns

- `.assignments` — a dict `{virtual_reg_id: physical_reg_id}`
- `.spilled` — virtual regs that were sent to memory
- `.copies_remaining` — `COPY`s whose source and destination ended up in
  *different* physregs (a coalescing-quality measure)

In [13]:
def allocate(
    build: Callable[
        [mir.MirModule], tuple[mir.MachineFunction, tuple[mir.Register, ...]]
    ],
    regalloc: str = "greedy",
    fn: str = "add",
) -> tuple[dict[int, str | int], list[int], int]:
    with ir.Context() as ctx:
        mod = ir.Module("m", ctx)
        tm = jit.TargetMachine(triple=TRIPLE)
        mmi = mir.create_machine_function(mod, tm, fn)
        mf, _ = build(mmi)
        names = phys_name_map(mf)
        res = mmi.regalloc_assignments(regalloc=regalloc)
        asg = {reg_name(vr): names.get(pr, pr) for vr, pr in res.assignments.items()}
        return asg, list(res.spilled), res.copies_remaining


asg, spilled, copies = allocate(build_add, "greedy")
print("assignments (vreg -> physreg):", asg)
print("spilled:", spilled)
print("copies remaining:", copies)

assignments (vreg -> physreg): {'%0': 'W0', '%1': 'W1', '%2': 'W0'}
spilled: []
copies remaining: 0


### Reflect

- All three vregs got a physical register and nothing spilled. Notice which
  physregs they landed in. The result copy `W0 = COPY %2` disappears if `%2` is
  already in `W0` — that's **coalescing**, and it's why `copies_remaining` is 0.
- Try changing `"greedy"` to `"basic"` (a much simpler first-free allocator).
  Do you get the same coloring here? Why might two allocators agree on an easy
  function but diverge on a hard one?

<details>
<summary><b>Answers</b> (click to expand)</summary>

- **`basic` vs `greedy` here:** on this pressure-free function they produce the
  same coloring — there's no real contention, so first-free and greedy both
  trivially succeed.
- **Why they'd diverge on a hard function:** two allocators agree when no choice
  matters (few candidates, no register pressure). They diverge once heuristics
  kick in — *which* value to evict, *where* to split a live range, *which* copies
  to coalesce. Those only bite under pressure, which is why an easy function
  hides the difference.

</details>

## 4. Live intervals and interference

Two virtual registers **interfere** if they are ever *live at the same time* —
then they cannot share a physical register. LLVM represents a value's lifetime
as a **live interval**: a set of `[start, end)` *segments* over `SlotIndex`
positions (numbered program points).

We can't inspect intervals from outside the allocator — they only exist while
the allocation passes are running. So we subclass the allocator and peek from
*inside* `select_or_split`, the per-interval callback. (We delegate the actual
decision to the built-in first-free allocator so the function still allocates.)

In [14]:
class Inspector(mir.BasicRegAlloc):
    def select_or_split(self, li: mir.LiveInterval) -> int | None:
        zero = self.zero_slot_index()
        segs = [(zero.distance(s.start), zero.distance(s.end)) for s in li.segments()]
        print(
            f"vreg {reg_name(li.reg)}: segments={segs}  weight={li.weight:.4g}  "
            f"spillable={li.is_spillable}"
        )
        return super().select_or_split(li)


mir.register_regalloc("inspect", Inspector)
with ir.Context() as ctx:
    mod = ir.Module("m", ctx)
    tm = jit.TargetMachine(triple=TRIPLE)
    mmi = mir.create_machine_function(mod, tm, "add")
    build_add(mmi)
    mmi.regalloc_assignments(regalloc="inspect")

vreg %1: segments=[(34, 50)]  weight=inf  spillable=False
vreg %2: segments=[(50, 66)]  weight=inf  spillable=False
vreg %0: segments=[(18, 50)]  weight=0.004676  spillable=True


Each vreg's segment `[start, end)` is its lifetime in slot-index units. Two
vregs interfere when their segments overlap. For our `add`, the two argument
copies are both live going into the `ADD`, so **they interfere** and must get
different registers — exactly what you saw in the assignment.

### Reflect

- Look at the printed segments. Which pairs overlap? Does that match which vregs
  received distinct physregs in Section 3?
- The `weight` field is a *spill cost* estimate (higher = more expensive to
  spill, e.g. used in a hot loop). Where do you think the allocator gets that
  number, and why would it matter when registers run out?

<details>
<summary><b>Answers</b> (click to expand)</summary>

- **Which overlap:** the two argument copies (`%0`, `%1`) are both live going
  into the `ADD`, so they overlap and must get **different** physregs — matching
  the distinct assignments in Section 3. The result `%2` is defined *at* the
  `ADD`, after `%0`/`%1` die, so it doesn't overlap them and can reuse one of
  their registers (e.g. `W0`).
- **Where `weight` comes from:** it's a spill-cost estimate — use/def frequency
  weighted by loop-nesting depth (block frequency), normalized by the interval's
  size (LLVM's `CalcSpillWeights`/`VirtRegAuxInfo`). It matters because under
  pressure the allocator spills the **cheapest** (lowest-weight) intervals first,
  keeping hot-loop values in registers.

</details>

<details>
<summary><b>Aside: when is a vreg <em>spillable</em>?</b> (click to expand)</summary>

`li.is_spillable` is just `weight != +inf`. A vreg *starts* spillable; LLVM's
`CalcSpillWeights` (`weightCalcHelper`) marks it **not** spillable (weight → inf)
in exactly three cases:

1. **Defined by an "unspillable terminator"** — a value produced into a register
   that a block terminator (e.g. a return) must read directly, so it cannot live
   on the stack. (This one does *not* apply to our `add` — none of `%0`/`%1`/`%2`
   is defined by the terminator.)
2. **A zero-length live range** — *unless* it is live across a call's
   register-mask clobber (or is a statepoint / mem-foldable inline-asm operand),
   where spilling may still be forced. **This is the case in our `add`**, but the
   segment numbers need decoding first, because "zero-length" is *not*
   `end - start == 0`.

   The numbers in a segment like `%1=[34,50)` are `SlotIndex` **positions**, not
   counts. Two facts about `SlotIndex`:

   - Instructions are numbered with a **fixed stride** — `InstrDist = 4 *
     Slot_Count = 16` (the four sub-slots below, times 4 for headroom) — **leaving
     gaps** so LLVM can *insert* spills/reloads/splits cheaply. In our function
     the instructions land at slots `COPY(%0 def)=16`, `COPY(%1 def)=32`,
     `ADDWrr(%2 def)=48`, `COPY(ret)=64`, `RET=80`. The gaps don't *avoid*
     renumbering, they make it *rare*: an insertion takes the **midpoint** of the
     surrounding gap (`dist = ((next-prev)/2) & ~3`), so each insertion into the
     same gap halves it; when `dist` rounds to 0 there's no slot left and LLVM
     runs a **local `renumberIndexes`** that re-spaces just the crowded region
     (with a global repack if a single renumber touches >20% of the function). So
     16 gives roughly three insertions of slack per gap before a renumber — an
     optimization, not a guarantee.
   - Each instruction owns a few **sub-slots**; a normal register def/use is at
     `base + 2`. So `%1=[34,50)` = "defined at the register slot of the
     instruction at base 32, live until its use at base 48."

   `LiveRange::isZeroLength` doesn't subtract; it asks *is any instruction
   strictly between def and use?* — `getNextNonNullIndex(start).getBaseIndex() <
   end.getBaseIndex()`. For our segments (`%0=[18,50)`, `%1=[34,50)`,
   `%2=[50,66)`):

   - `%1`: def at base 32, use at base 48 — **consecutive** instructions, nothing
     in between → zero-length → unspillable. (Its width `50-34=16` is exactly one
     instruction stride, i.e. def and use are adjacent.)
   - `%2`: def `ADD`(48), use ret-`COPY`(64) — also adjacent → zero-length.
   - `%0`: def `COPY`(16), use `ADD`(48), and the `COPY` at base **32 sits
     strictly between** → **not** zero-length → spillable.

   So "zero-length" means **there is no slot between def and use to place spill
   code**, not that the range is empty. The distinguishing factor is whether an
   instruction lies in the gap, not which value reaches the return: define `%1`
   first (so the other copy falls inside its range) and `%1` becomes spillable
   while the now-adjacent `%0` becomes unspillable.
3. **Split off from an already-unspillable interval** — unspillability is
   inherited by split children, and once set it is never recomputed back.

So a vreg is **spillable** when none of those hold: it isn't a terminator-defined
value, it has a real (non-trivial) live range, and it didn't descend from an
unspillable interval. Two related non-conditions: a *rematerializable* vreg is
still spillable (its weight is merely *halved* to make it a preferred spill
candidate — recomputing beats a stack round-trip), and a *dead* vreg isn't
"unspillable", it's simply never allocated.

This is why `RAILPDecomp` **force-assigns** unspillable vregs (never puts them in
its spill set): asking the framework to spill one makes LLVM's spiller abort, so
the base guards `self.spill(li)` with `li.is_spillable`.

</details>

## 5. The allocation framework: `select_or_split`

LLVM's allocators (and ours) share a driver that processes one virtual register
at a time, in a priority order, calling `select_or_split(live_interval)`. The
override must do one of:

- **return a physical register id** → assign the vreg to it, or
- **call `self.spill(li)`** → send it to memory (this creates small new
  *reload* vregs that get re-queued), or
- split the range (advanced; greedy does this).

The simplest possible allocator: for each vreg, take the first physical register
in the target's preferred order that is currently free; if none is free, spill.
That's exactly the built-in `BasicRegAlloc`. Here it is, in full:

In [15]:
import inspect

print(inspect.getsource(mir.BasicRegAlloc))

class BasicRegAlloc(mir.RegAllocBase):
    """First-free-or-spill allocator.

    Relies on the C++ default spill-weight queue (no enqueue/dequeue override):
    for each unassigned live interval, take the first interference-free physreg
    in the target allocation order, else spill it and let the resulting split
    vregs be re-enqueued.
    """

    def select_or_split(self, li):
        for preg in self.allocation_order(li):
            if self.matrix.is_free(li, preg):
                return preg
        self.spill(li)
        return None



Two framework services appear here:

- `self.allocation_order(li)` — the physical registers legal for this vreg's
  register class, already ordered by preference (copy-hints first, then the
  target's allocation order).
- `self.matrix.is_free(li, preg)` — asks the **live-register matrix** whether
  assigning `preg` to `li` would clash with anything already assigned.

Let's watch it choose, printing the candidates it considers and the one it picks.

In [16]:
class VerboseBasic(mir.BasicRegAlloc):
    def select_or_split(self, li: mir.LiveInterval) -> int | None:
        names = self._names
        order = self.allocation_order(li)
        chosen = None
        for preg in order:
            if self.matrix.is_free(li, preg):
                chosen = preg
                break
        shown = ", ".join(names.get(p, str(p)) for p in list(order)[:6])
        print(
            f"vreg {reg_name(li.reg)}: order=[{shown}, ...]  -> picked "
            f"{names.get(chosen, chosen)}"
        )
        return super().select_or_split(li)


mir.register_regalloc("verbose-basic", VerboseBasic)
with ir.Context() as ctx:
    mod = ir.Module("m", ctx)
    tm = jit.TargetMachine(triple=TRIPLE)
    mmi = mir.create_machine_function(mod, tm, "add")
    mf, _ = build_add(mmi)
    VerboseBasic._names = phys_name_map(mf)
    mmi.regalloc_assignments(regalloc="verbose-basic")

vreg %1: order=[W1, W1, W8, W9, W10, W11, ...]  -> picked W1
vreg %2: order=[W0, W0, W8, W9, W10, W11, ...]  -> picked W0
vreg %0: order=[W0, W0, W8, W9, W10, W11, ...]  -> picked W0


### Reflect

- The order starts with the same physreg for several vregs, yet they end up
  distinct. What changed between the first and second vreg that made
  `is_free` reject the earlier choice?
- `BasicRegAlloc` never *evicts* an already-assigned register or *splits* a
  range — it only assigns-or-spills. What kinds of programs would make this
  allocator spill unnecessarily compared to a smarter one?

<details>
<summary><b>Answers</b> (click to expand)</summary>

- **What changed between the 1st and 2nd vreg:** once the first vreg is assigned,
  the live-register **matrix** records that physreg as occupied over its range.
  The second vreg is live at the same point (they interfere), so `is_free`
  rejects that physreg and the loop falls through to the next candidate.
- **When `basic` spills needlessly:** whenever the blocking register holds a
  *cheaper, evictable* virtual register (greedy would kick it out), or when a
  long range could be **split** so the part near its uses stays in a register and
  only the pass-through part spills. `basic` can do neither, so it spills the
  whole value where a smarter allocator wouldn't.

</details>

## 6. The interference matrix, up close

`matrix.check_interference(li, preg)` returns a *kind*, not just a bool:

| Kind | Meaning |
|------|---------|
| `IK_Free` | nothing conflicts — you may assign `preg` |
| `IK_VirtReg` | another **virtual** reg is assigned here (could be *evicted*) |
| `IK_RegUnit` | a **fixed physical** reg / sub-register conflict |
| `IK_RegMask` | a call clobbers `preg` across this range |

This distinction is what lets `greedy` do more than `basic`: an `IK_VirtReg`
clash is *negotiable* (kick the other guy out), while `IK_RegUnit`/`IK_RegMask`
are hard constraints.

In [17]:
class MatrixPeek(mir.BasicRegAlloc):
    def select_or_split(self, li: mir.LiveInterval) -> int | None:
        names = getattr(self, "_names", {})
        counts = {}
        for preg in self.allocation_order(li):
            kind = self.matrix.check_interference(li, preg)
            counts[kind.name] = counts.get(kind.name, 0) + 1
        print(
            f"vreg {reg_name(li.reg)}: interference kinds across candidates -> {counts}"
        )
        return super().select_or_split(li)


mir.register_regalloc("matrix-peek", MatrixPeek)
with ir.Context() as ctx:
    mod = ir.Module("m", ctx)
    tm = jit.TargetMachine(triple=TRIPLE)
    mmi = mir.create_machine_function(mod, tm, "add")
    mf, _ = build_add(mmi)
    MatrixPeek._names = phys_name_map(mf)
    mmi.regalloc_assignments(regalloc="matrix-peek")

vreg %1: interference kinds across candidates -> {'IK_Free': 31}
vreg %2: interference kinds across candidates -> {'IK_Free': 31}
vreg %0: interference kinds across candidates -> {'IK_Free': 30, 'IK_RegUnit': 1}


## 7. Spilling and reloads (register pressure)

When more values are simultaneously live than there are registers, some must go
to memory — a **spill**. Its uses are then served by loading it back into a
register just before each use — a **reload**. LLVM's spiller does this by
rewriting the spilled vreg's uses to small fresh *reload* vregs.

Let's create pressure: 48 values all live at once (more than the ~31 allocatable
GPR32s), then combined. Greedy must spill.

In [18]:
def build_high_pressure(
    mmi: mir.MirModule, n: int = 48
) -> tuple[mir.MachineFunction, tuple[mir.Register, ...]]:
    mf = mmi.machine_function("hp")
    b = mir.MachineIRBuilder(mf)
    gpr32 = mf.reg_class("GPR32")
    w0 = mf.physreg("W0")
    mf.blocks[0].add_livein(w0)
    copy, addrr = mf.opcode("COPY"), mf.opcode("ADDWrr")
    terms = []
    for _ in range(n):
        t = mf.create_vreg(gpr32)
        ins = b.build_instr(copy)
        ins.add_reg(t, is_def=True)
        ins.add_reg(w0)
        terms.append(t)
    acc = terms[0]
    for t in terms[1:]:
        nacc = mf.create_vreg(gpr32)
        ins = b.build_instr(addrr)
        ins.add_reg(nacc, is_def=True)
        ins.add_reg(acc)
        ins.add_reg(t)
        acc = nacc
    rc = b.build_instr(copy)
    rc.add_reg(w0, is_def=True)
    rc.add_reg(acc)
    b.build_instr(mf.opcode("RET_ReallyLR")).add_reg(w0, implicit=True)
    for prop in ("IsSSA", "TracksLiveness", "NoPHIs"):
        mf.set_property(getattr(mir.MachineFunctionProperty, prop))
    return mf, ()


asg, spilled, copies = allocate(build_high_pressure, "greedy", fn="hp")
print(
    f"assigned {len(asg)} vregs; spilled {len(spilled)} vregs; "
    f"{copies} copies remaining"
)
print("a few assignments:", dict(list(asg.items())[:5]))

assigned 95 vregs; spilled 20 vregs; 27 copies remaining
a few assignments: {'%3': 'W10', '%4': 'W11', '%5': 'W12', '%6': 'W13', '%7': 'W14'}


### Reflect

- Greedy assigned most values and spilled the rest. The number of allocatable
  GPR32 registers is around 31 — does the spill count line up with
  "48 minus the registers"? (Careful: a spilled value still needs a register
  *momentarily* at each use — a reload. That subtlety comes back in Section 9.)
- The spiller turns one spilled vreg into several tiny reload vregs. Why does
  that make "how many registers do I really need?" harder than just counting the
  original values?

<details>
<summary><b>Answers</b> (click to expand)</summary>

- **Does spill count = 48 − 31?** Not exactly. The naive bound undercounts
  because **reloads** reintroduce register demand at each use, and the add-chain
  creates intermediate accumulator values too — so you see a few *more* spills
  (~20) than "48 minus the registers".
- **Why reloads make it hard:** spilling one value doesn't remove its register
  need everywhere — at each use it must be loaded back into a register (a short
  reload live range), and its def written from one. So the true register demand
  isn't the original simultaneously-live count; it's a fixed point that depends
  on where the reloads land. (This is the exact trap that sinks two of the ILP
  allocators in Section 9.)

</details>

## 8. What `greedy` actually does: evict and split

`BasicRegAlloc` only assigns-or-spills. LLVM's `greedy` has two more moves:

- **Eviction**: if the only candidates are blocked by *other virtual registers*
  (`IK_VirtReg`) that are cheaper (lower spill weight), kick them out, take the
  register, and re-queue the evicted ones.
- **Live-range splitting**: cut one long interval into pieces so part of it can
  stay in a register while the rest spills — far less costly than spilling the
  whole thing.

A clarification first: `regalloc="greedy"` (Sections 3 and 9) dispatches to
LLVM's **built-in C++** greedy allocator — *no Python runs* during it.
Separately, `eudsl-llvmpy` ships a faithful Python **reimplementation**,
`mir.RAGreedy`, whose `select_or_split` runs in Python on every interval — which
is exactly what lets us trace it. It records what it did per vreg in a `trace`
dict (`assign` / `evict` / `region_split` / `block_split` / `local_split` /
`spill`). Let's capture it.

In [19]:
_TRACE = {}


class TracedGreedy(mir.RAGreedy):
    def select_or_split(self, li: mir.LiveInterval) -> int | None:
        r = super().select_or_split(li)
        _TRACE.update(self.trace)
        return r


mir.register_regalloc("traced-greedy", TracedGreedy)
_TRACE.clear()
with ir.Context() as ctx:
    mod = ir.Module("m", ctx)
    tm = jit.TargetMachine(triple=TRIPLE)
    mmi = mir.create_machine_function(mod, tm, "hp")
    build_high_pressure(mmi)
    mmi.regalloc_assignments(regalloc="traced-greedy")

from collections import Counter

print("actions taken by greedy on the high-pressure function:")
print(Counter(_TRACE.values()))

actions taken by greedy on the high-pressure function:
Counter({'assign': 92, 'spill': 20, 'evict': 3})


### Reflect

- Which actions dominate? On this straight-line function you'll see mostly
  `assign` and `spill`; splitting shines more on code with loops and branches.
- Eviction and splitting are *heuristics* — fast, local decisions. That raises a
  natural question: how far from *optimal* is greedy? To answer that, you need
  an allocator that reasons about the whole function at once.

<details>
<summary><b>Answers</b> (click to expand)</summary>

- **Which actions dominate:** on this straight-line function, `assign` and
  `spill`, with a *few* `evict`s (the printed `Counter` shows ~`assign: 92,
  spill: 20, evict: 3`). Splitting mostly pays off across loops/branches, which
  this fixture doesn't have.
- **How far from optimal is greedy?** It's a heuristic — no optimality
  guarantee. The only way to *know* the gap is to compare against an allocator
  that optimizes the whole function at once. That's precisely the ILP allocators
  in Section 9 (which even report their optimality `gap`).

</details>

## 9. Global allocation: the ILP allocators

`eudsl-llvmpy` also includes three allocators that formulate allocation as a
global optimization problem solved with an ILP / **CP-SAT** (constraint
programming over a **SAT**, i.e. boolean-satisfiability, solver) engine from Google **OR-Tools** (Operations
Research Tools):

- `mir.RAILPAssign` — the classic 0-1 ILP: a boolean `x[vreg, preg]` per legal
  pair plus a spill variable, "exactly one" per vreg, interference edges, and an
  objective minimizing weighted spill cost (with a coalescing bonus).
- `mir.RAILPPacking` — models each vreg as a rectangle on a (time × register)
  grid and forbids overlap (a 2D bin-packing view).
- `mir.RAILPDecomp` — spill *then* color: a per-program-point ILP chooses the
  minimum-weight spill set, then colors what remains.

On an easy (register-fitting) function all three fit everything in registers
(no spills) and prove they're **optimal for their model** (`gap 0`). They need
not be *identical*, though — coalescing differs, so e.g. `RAILPPacking` may leave
one copy that `RAILPAssign`/`RAILPDecomp` remove.

In [20]:
from llvm.mir_ilp_base import RAILPBase


def run_ilp(
    build: Callable[
        [mir.MirModule], tuple[mir.MachineFunction, tuple[mir.Register, ...]]
    ],
    name: str,
    cls: type[RAILPBase],
    fn: str,
) -> str:
    mir.register_regalloc(name, cls)
    try:
        asg, spilled, copies = allocate(build, name, fn=fn)
        stats = RAILPBase.last_stats.get(cls.__name__)
        gap = None if stats is None else stats.gap
        return f"valid, spills={len(spilled)}, copies={copies}, gap={gap}"
    except RuntimeError as e:
        return "hard-fail: " + str(e).split(";")[0]


for nm, cls in [
    ("a", mir.RAILPAssign),
    ("p", mir.RAILPPacking),
    ("d", mir.RAILPDecomp),
]:
    print(f"{cls.__name__:14} ->", run_ilp(build_add, "ilp-" + nm, cls, "add"))

RAILPAssign    -> valid, spills=0, copies=0, gap=0.0
RAILPPacking   -> valid, spills=0, copies=1, gap=0.0
RAILPDecomp    -> valid, spills=0, copies=0, gap=0.0


Now the hard case. Here the three diverge, and the reason is the deepest lesson
in this notebook.

`RAILPAssign` and `RAILPPacking` decide spills at *whole-live-interval*
granularity — "spill this value entirely or not at all." That view **ignores
reload pressure**: a spilled value still needs a register at each use. Their
minimum-spill solutions can therefore be *un-realizable*, and the framework
would be asked to do something impossible — so they **hard-fail cleanly** rather
than emit wrong code.

`RAILPDecomp` uses the per-program-point (Appel–George) model: at every def/use
point it counts a spilled value as still needing a register there. That makes
its spill set realizable, so it **succeeds** where the other two decline.

In [21]:
for nm, cls in [
    ("a", mir.RAILPAssign),
    ("p", mir.RAILPPacking),
    ("d", mir.RAILPDecomp),
]:
    print(
        f"{cls.__name__:14} ->", run_ilp(build_high_pressure, "ilp-hp-" + nm, cls, "hp")
    )

RAILPAssign    -> hard-fail: RAILPAssign does not realize spills, but the ILP requires spilling 18 vreg(s)
RAILPPacking   -> hard-fail: RAILPPacking does not realize spills, but the ILP requires spilling 18 vreg(s)
RAILPDecomp    -> valid, spills=19, copies=28, gap=0.0


### Reflect

- `RAILPDecomp` allocated the high-pressure function; the other two hard-failed.
  In your own words, *why* is "spill the whole value" not enough — what does a
  spilled value still cost at each use?
- This mirrors the compiler literature exactly: optimal *spilling* is the hard
  part, and it must be modeled per program point (Appel & George, 2001), not per
  whole interval.
- `greedy` handled the same function without hard-failing. What tool does greedy
  have (Section 8) that the whole-interval ILP models lack, which lets it get
  away with a cruder spill model?

<details>
<summary><b>Answers</b> (click to expand)</summary>

- **Why "spill the whole value" isn't enough:** a spilled value still needs a
  register *momentarily* at every def/use — the store source and each reload. A
  whole-interval model (`RAILPAssign`/`RAILPPacking`) counts a spilled value as
  needing **zero** registers, so it undercounts pressure exactly at those points
  and can produce an *unrealizable* spill set — hence the clean hard-fail.
- **What the per-point model fixes:** `RAILPDecomp` counts a spilled value as
  still occupying a register at each def/use point (the Appel-George model), so
  its spill set is always realizable — which is why it succeeds where the other
  two decline.
- **What greedy has that they lack:** **live-range splitting** (plus eviction and
  iteration). Greedy can split a long range so only the pass-through portion
  spills while the parts near uses stay in registers — sidestepping the
  reload-pressure problem that a crude whole-interval spill model walks straight
  into.

</details>

## Where to go next

- Add a loop to a fixture and re-run the `traced-greedy` cell: do you see
  `region_split` / `block_split` appear?
- Compare `copies_remaining` across `greedy`, `RAILPAssign`, and `RAILPDecomp`
  on a copy-heavy function — which coalesces best?
- Read `select_or_split` in `mir.RAGreedy` (`inspect.getsource(mir.RAGreedy)`)
  and trace how it escalates: assign → evict → split → spill.